In [2]:
#!/usr/bin/env python3
"""
Movie Recommendation System
===========================

Three techniques in one small, dependency-light module (numpy, pandas, scikit-learn):

  user_cf   User-based collaborative filtering.
            "People whose ratings look like yours liked these."
  item_cf   Item-based collaborative filtering.
            "Movies rated similarly to the ones you liked."
  content   Content-based filtering on genres (TF-IDF + cosine similarity).
            "Movies that share genres with the ones you liked."
  hybrid    Weighted blend of the three above (falls back gracefully when one
            method cannot score a movie).

Cold start (a brand-new user with no ratings) is handled by genre queries and a
popularity ranking.

Usage
-----
  python recommender.py                              # full demo on built-in sample data
  python recommender.py --user 3 --method hybrid     # recommendations for user 3
  python recommender.py --similar "Inception"        # movies similar to a title
  python recommender.py --genres "Sci-Fi,Thriller"   # cold start: pick by genre
  python recommender.py --rate "Inception=5;Titanic=2;Alien=4"   # new user on the fly
  python recommender.py --evaluate                   # hold-out RMSE / MAE / precision
  python recommender.py --movies movies.csv --ratings ratings.csv ...   # MovieLens format
"""
from __future__ import annotations

import argparse
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

METHODS = ["user_cf", "item_cf", "content", "hybrid"]
HYBRID_WEIGHTS = {"user_cf": 0.30, "item_cf": 0.30, "content": 0.40}


# --------------------------------------------------------------------------- #
# Sample data (synthetic ratings with clear taste groups, so results are
# easy to sanity-check). Replace with MovieLens via --movies / --ratings.
# --------------------------------------------------------------------------- #
MOVIES = [
    (1, "The Shawshank Redemption (1994)", "Drama"),
    (2, "The Godfather (1972)", "Crime|Drama"),
    (3, "The Dark Knight (2008)", "Action|Crime|Drama"),
    (4, "Pulp Fiction (1994)", "Crime|Drama|Thriller"),
    (5, "Forrest Gump (1994)", "Comedy|Drama|Romance"),
    (6, "Inception (2010)", "Action|Sci-Fi|Thriller"),
    (7, "The Matrix (1999)", "Action|Sci-Fi"),
    (8, "Interstellar (2014)", "Adventure|Drama|Sci-Fi"),
    (9, "Toy Story (1995)", "Animation|Children|Comedy"),
    (10, "Finding Nemo (2003)", "Animation|Children|Comedy"),
    (11, "The Lion King (1994)", "Animation|Children|Drama"),
    (12, "Spirited Away (2001)", "Animation|Fantasy|Adventure"),
    (13, "Titanic (1997)", "Drama|Romance"),
    (14, "The Notebook (2004)", "Drama|Romance"),
    (15, "When Harry Met Sally (1989)", "Comedy|Romance"),
    (16, "Pretty Woman (1990)", "Comedy|Romance"),
    (17, "Mean Girls (2004)", "Comedy"),
    (18, "Superbad (2007)", "Comedy"),
    (19, "The Hangover (2009)", "Comedy"),
    (20, "Die Hard (1988)", "Action|Thriller"),
    (21, "Mad Max: Fury Road (2015)", "Action|Adventure|Sci-Fi"),
    (22, "Gladiator (2000)", "Action|Adventure|Drama"),
    (23, "Raiders of the Lost Ark (1981)", "Action|Adventure"),
    (24, "Jurassic Park (1993)", "Action|Adventure|Sci-Fi|Thriller"),
    (25, "Alien (1979)", "Horror|Sci-Fi"),
    (26, "The Shining (1980)", "Horror"),
    (27, "Get Out (2017)", "Horror|Mystery|Thriller"),
    (28, "Hereditary (2018)", "Horror|Mystery"),
    (29, "Se7en (1995)", "Crime|Mystery|Thriller"),
    (30, "Gone Girl (2014)", "Drama|Mystery|Thriller"),
    (31, "The Silence of the Lambs (1991)", "Crime|Horror|Thriller"),
    (32, "The Fellowship of the Ring (2001)", "Adventure|Fantasy"),
    (33, "Harry Potter and the Sorcerer's Stone (2001)", "Adventure|Children|Fantasy"),
    (34, "Pan's Labyrinth (2006)", "Drama|Fantasy|War"),
    (35, "Saving Private Ryan (1998)", "Action|Drama|War"),
    (36, "Schindler's List (1993)", "Drama|War"),
    (37, "La La Land (2016)", "Comedy|Drama|Musical|Romance"),
    (38, "The Princess Bride (1987)", "Adventure|Comedy|Fantasy|Romance"),
    (39, "Blade Runner 2049 (2017)", "Drama|Mystery|Sci-Fi"),
    (40, "Parasite (2019)", "Comedy|Drama|Thriller"),
]

# Hidden genre affinities (-1..1) used only to *generate* the sample ratings.
PERSONAS = {
    "action_fan": {"Action": .9, "Adventure": .6, "Thriller": .4, "Sci-Fi": .3, "Romance": -.5, "Children": -.3},
    "scifi_fan": {"Sci-Fi": .9, "Mystery": .4, "Thriller": .3, "Adventure": .3, "Romance": -.3},
    "romance_fan": {"Romance": .9, "Comedy": .5, "Musical": .6, "Drama": .3, "Horror": -.6, "War": -.4, "Action": -.3},
    "drama_fan": {"Drama": .9, "Crime": .4, "War": .4, "Mystery": .3, "Comedy": -.1, "Children": -.2, "Horror": -.4},
    "family_fan": {"Animation": .9, "Children": .9, "Fantasy": .6, "Adventure": .5, "Comedy": .3, "Horror": -.8, "Crime": -.4},
    "horror_fan": {"Horror": .9, "Mystery": .6, "Thriller": .6, "Crime": .3, "Romance": -.5, "Children": -.6},
    "comedy_fan": {"Comedy": .9, "Romance": .3, "Animation": .3, "Drama": -.2, "War": -.4, "Horror": -.3},
}
USERS_PER_PERSONA = 6


def build_sample_data(seed: int = 7):
    """Return (movies, ratings, {userId: persona}) with reproducible synthetic ratings."""
    rng = np.random.default_rng(seed)
    movies = pd.DataFrame(MOVIES, columns=["movieId", "title", "genres"])
    genre_lists = movies["genres"].str.split("|")
    all_genres = sorted({g for gl in genre_lists for g in gl})
    quality = rng.normal(0, 0.3, len(movies))  # some films are just better liked by everyone

    rows, personas, uid = [], {}, 1
    for persona, affinity in PERSONAS.items():
        for _ in range(USERS_PER_PERSONA):
            taste = {g: affinity.get(g, 0.0) + rng.normal(0, 0.15) for g in all_genres}
            bias = rng.normal(0, 0.3)
            mean_aff = np.array([np.mean([taste[g] for g in gl]) for gl in genre_lists])
            p_seen = np.clip(0.35 + 0.35 * mean_aff, 0.08, 0.8)  # people watch what they like
            seen = rng.random(len(movies)) < p_seen
            if seen.sum() < 8:
                seen[rng.choice(len(movies), 8, replace=False)] = True
            raw = 3.2 + 1.6 * mean_aff + quality + bias + rng.normal(0, 0.4, len(movies))
            stars = np.clip(np.round(raw), 1, 5)
            for mid, s in zip(movies["movieId"][seen], stars[seen]):
                rows.append((uid, int(mid), float(s)))
            personas[uid] = persona
            uid += 1
    ratings = pd.DataFrame(rows, columns=["userId", "movieId", "rating"])
    return movies, ratings, personas


# --------------------------------------------------------------------------- #
# The recommender
# --------------------------------------------------------------------------- #
class Recommender:
    def __init__(self, movies: pd.DataFrame, ratings: pd.DataFrame,
                 k_users: int = 40, k_items: int = 20, shrink: int = 5):
        """
        k_users / k_items : number of nearest neighbours used for a prediction.
        shrink            : similarities built from fewer than `shrink` co-rated
                            entries are scaled down (significance weighting),
                            which stops two users who share a single movie from
                            looking like "twins".
        """
        self.movies = movies.drop_duplicates("movieId").set_index("movieId").sort_index()
        self.k_users, self.k_items, self.shrink = k_users, k_items, shrink
        self.lo, self.hi = float(ratings["rating"].min()), float(ratings["rating"].max())

        # user x movie matrix (NaN = not rated); every catalogue movie gets a column
        self.R = (ratings.pivot_table(index="userId", columns="movieId", values="rating")
                  .reindex(columns=self.movies.index))
        self.users = self.R.index
        self.mask = self.R.notna().to_numpy().astype(float)

        self.global_mean = float(ratings["rating"].mean())
        self.user_mean = self.R.mean(axis=1)
        self.item_mean = self.R.mean(axis=0)  # NaN for movies nobody rated

        # Bayesian ("true") average: pulls movies with few ratings toward the global mean
        counts, sums = self.R.notna().sum(axis=0), self.R.sum(axis=0)
        m = 5
        self.popularity = (sums + m * self.global_mean) / (counts + m)

        # Mean-centred ratings remove "this user rates everything high/low" bias.
        self.C = self.R.sub(self.user_mean, axis=0).fillna(0.0).to_numpy()

        # --- collaborative similarities (cosine on centred ratings ~ Pearson) ---
        self.user_sim = self._shrunk(cosine_similarity(self.C), self.mask @ self.mask.T)
        self.item_sim = self._shrunk(cosine_similarity(self.C.T), self.mask.T @ self.mask)

        # --- content similarity from genres (TF-IDF down-weights very common genres) ---
        self.tfidf = TfidfVectorizer(tokenizer=lambda s: s.split("|"), lowercase=False, token_pattern=None)
        self.G = self.tfidf.fit_transform(self.movies["genres"])
        self.content_sim = cosine_similarity(self.G)
        np.fill_diagonal(self.content_sim, 0.0)
        self.genres = sorted(self.tfidf.vocabulary_)

    def _shrunk(self, sim: np.ndarray, common: np.ndarray) -> np.ndarray:
        sim = sim * np.minimum(common, self.shrink) / self.shrink
        np.fill_diagonal(sim, 0.0)
        return sim

    # ----------------------------- predictors ------------------------------ #
    def _user_cf(self, u: int) -> np.ndarray:
        s = self.user_sim[u].copy()
        s[s < 0] = 0.0
        if self.k_users < len(s):
            s[s < np.partition(s, -self.k_users)[-self.k_users]] = 0.0
        num, den = s @ self.C, s @ self.mask
        with np.errstate(divide="ignore", invalid="ignore"):
            pred = self.user_mean.iloc[u] + num / den
        pred[den == 0] = np.nan
        return pred

    def _item_cf(self, u: int, sim: np.ndarray) -> np.ndarray:
        """Predict from the movies user u rated, weighted by similarity to each target movie."""
        rated = np.flatnonzero(self.mask[u])
        if rated.size == 0:
            return np.full(sim.shape[0], np.nan)
        S = sim[:, rated].copy()
        S[S < 0] = 0.0
        k = min(self.k_items, rated.size)
        if k < S.shape[1]:
            S[S < np.partition(S, -k, axis=1)[:, -k][:, None]] = 0.0
        num, den = S @ self.C[u, rated], S.sum(axis=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            pred = self.user_mean.iloc[u] + num / den
        pred[den == 0] = np.nan
        return pred

    def predict(self, user_id, method: str = "hybrid") -> pd.Series:
        """Predicted rating for every catalogue movie (NaN where the method cannot score it)."""
        if user_id not in self.users:
            raise KeyError(f"unknown user {user_id!r}")
        u, n = self.users.get_loc(user_id), len(self.movies)
        if method == "global_mean":
            vals = np.full(n, self.global_mean)
        elif method == "user_mean":
            vals = np.full(n, self.user_mean.iloc[u])
        elif method == "item_mean":
            vals = self.item_mean.to_numpy()
        elif method == "user_cf":
            vals = self._user_cf(u)
        elif method == "item_cf":
            vals = self._item_cf(u, self.item_sim)
        elif method == "content":
            vals = self._item_cf(u, self.content_sim)
        elif method == "hybrid":
            P = np.vstack([self.predict(user_id, m).to_numpy() for m in HYBRID_WEIGHTS])
            W = np.array(list(HYBRID_WEIGHTS.values()))[:, None] * ~np.isnan(P)
            with np.errstate(divide="ignore", invalid="ignore"):
                vals = (np.nan_to_num(P) * W).sum(axis=0) / W.sum(axis=0)
        else:
            raise ValueError(f"unknown method {method!r}")
        return pd.Series(np.clip(vals, self.lo, self.hi), index=self.movies.index)

    # ----------------------------- recommendations ------------------------- #
    def _unseen(self, user_id) -> np.ndarray:
        return self.mask[self.users.get_loc(user_id)] == 0

    def top_ids(self, scores: pd.Series, user_id, n: int) -> list:
        s = scores[self._unseen(user_id)].dropna()
        return list(s.sort_values(ascending=False, kind="stable").head(n).index)

    def recommend(self, user_id, method: str = "hybrid", n: int = 10) -> pd.DataFrame:
        scores = self.predict(user_id, method)
        ids = self.top_ids(scores, user_id, n)
        if len(ids) < n:  # not enough signal -> top up with what's popular
            extra = self.top_ids(self.popularity, user_id, n + len(ids))
            ids += [i for i in extra if i not in ids][: n - len(ids)]
        out = self.movies.loc[ids, ["title", "genres"]].copy()
        out["predicted"] = [round(float(scores.get(i, self.popularity[i])), 2) for i in ids]
        return out.reset_index(drop=True)

    def history(self, user_id, n: int = 5) -> pd.DataFrame:
        r = self.R.loc[user_id].dropna().sort_values(ascending=False, kind="stable").head(n)
        out = self.movies.loc[r.index, ["title", "genres"]].copy()
        out["rating"] = r.to_numpy()
        return out.reset_index(drop=True)

    def similar_movies(self, movie_id: int, n: int = 5, by: str = "content") -> pd.DataFrame:
        sim = self.content_sim if by == "content" else self.item_sim
        row = pd.DataFrame({"sim": sim[self.movies.index.get_loc(movie_id)],
                            "pop": self.popularity.to_numpy()}, index=self.movies.index)
        row = row.drop(movie_id).sort_values(["sim", "pop"], ascending=False).head(n)
        out = self.movies.loc[row.index, ["title", "genres"]].copy()
        out["similarity"] = row["sim"].round(2).to_numpy()
        return out.reset_index(drop=True)

    def by_genres(self, genres: list[str], n: int = 10) -> pd.DataFrame:
        """Cold start: no ratings needed. Rank by genre match, then by popularity."""
        lookup = {g.lower(): g for g in self.genres}
        bad = [g for g in genres if g.lower() not in lookup]
        if bad:
            raise ValueError(f"unknown genre(s) {bad}; choose from {self.genres}")
        q = self.tfidf.transform(["|".join(lookup[g.lower()] for g in genres)])
        match = cosine_similarity(q, self.G).ravel()
        score = pd.Series(match * self.popularity.to_numpy(), index=self.movies.index)
        ids = list(score[match > 0].sort_values(ascending=False, kind="stable").head(n).index)
        out = self.movies.loc[ids, ["title", "genres"]].copy()
        out["match"] = match[[self.movies.index.get_loc(i) for i in ids]].round(2)
        return out.reset_index(drop=True)

    def find_movie(self, query: str) -> int:
        titles = self.movies["title"]
        exact = titles[titles.str.lower() == query.lower()]
        hits = exact if len(exact) else titles[titles.str.contains(query, case=False, regex=False)]
        if hits.empty:
            raise ValueError(f"no movie matches {query!r}")
        if len(hits) > 1:
            print(f"(note: {query!r} matched {len(hits)} titles, using {hits.iloc[0]!r})", file=sys.stderr)
        return int(hits.index[0])


# --------------------------------------------------------------------------- #
# Evaluation: hold out ~20% of each user's ratings, predict them, score.
# --------------------------------------------------------------------------- #
def evaluate(movies, ratings, repeats: int = 3, test_frac: float = 0.2,
             like: float = 4.0, top_n: int = 5, seed: int = 0) -> pd.DataFrame:
    methods = ["global_mean", "user_mean", "item_mean", "user_cf", "item_cf", "content", "hybrid"]
    ranking = {"item_mean", "user_cf", "item_cf", "content", "hybrid"}
    runs = []
    for r in range(repeats):
        test = ratings.groupby("userId").sample(frac=test_frac, random_state=seed + r)
        train = ratings.drop(test.index)
        rec = Recommender(movies, train)
        st = {m: {"err": [], "cov": [], "prec": []} for m in methods}
        for uid, t in test.groupby("userId"):
            if uid not in rec.users:
                continue
            fallback = rec.user_mean[uid]
            liked = set(t.loc[t["rating"] >= like, "movieId"])
            for m in methods:
                pred = rec.predict(uid, m)
                p = pred.reindex(t["movieId"]).to_numpy()
                covered = ~np.isnan(p)
                p = np.where(covered, p, fallback)  # unscorable -> user's average
                st[m]["err"].extend(p - t["rating"].to_numpy())
                st[m]["cov"].extend(covered)
                if m in ranking and liked:
                    top = rec.top_ids(pred, uid, top_n)
                    st[m]["prec"].append(len(liked & set(top)) / top_n)
        runs.append(pd.DataFrame({
            m: {"RMSE": float(np.sqrt(np.mean(np.square(v["err"])))),
                "MAE": float(np.mean(np.abs(v["err"]))),
                "coverage": float(np.mean(v["cov"])),
                f"P@{top_n}": float(np.mean(v["prec"])) if v["prec"] else np.nan}
            for m, v in st.items()}).T)
    res = sum(runs) / len(runs)
    res.loc[["global_mean", "user_mean"], f"P@{top_n}"] = np.nan  # constant -> no ranking
    res.index.name = f"method (avg of {repeats} splits)"
    return res.round(3)


# --------------------------------------------------------------------------- #
# CLI
# --------------------------------------------------------------------------- #
def show(title: str, df: pd.DataFrame):
    print(f"\n{title}")
    print(df.to_string(index=False))


def parse_ratings_arg(text: str, rec: Recommender) -> list[tuple[int, float]]:
    out = []
    for part in filter(None, (p.strip() for p in text.split(";"))):
        title, sep, val = part.rpartition("=")
        if not sep:
            raise ValueError(f"expected 'Title=rating', got {part!r}")
        out.append((rec.find_movie(title.strip()), float(val)))
    return out


def run_demo(rec: Recommender, movies, ratings, personas):
    n_u, n_m, n_r = len(rec.users), len(rec.movies), len(ratings)
    print(f"Dataset: {n_u} users, {n_m} movies, {n_r} ratings "
          f"({100 * n_r / (n_u * n_m):.0f}% of the user x movie matrix filled)")

    uid = int(rec.users[0])
    tag = f" (hidden taste group: {personas[uid]})" if personas else ""
    show(f"User {uid}{tag} - highest rated so far", rec.history(uid))
    show(f"User {uid} - HYBRID top 5", rec.recommend(uid, "hybrid", 5))
    for m in ("user_cf", "item_cf", "content"):
        show(f"User {uid} - {m.upper()} top 3", rec.recommend(uid, m, 3))

    mid = rec.find_movie("Inception")
    show("Similar to 'Inception' by GENRE (content-based)", rec.similar_movies(mid, 5, "content"))
    show("Similar to 'Inception' by RATING PATTERNS (item-based CF)", rec.similar_movies(mid, 5, "ratings"))

    show("Cold start: genres = Sci-Fi, Thriller (no ratings needed)", rec.by_genres(["Sci-Fi", "Thriller"], 5))

    print("\nEvaluating on held-out ratings (lower RMSE/MAE is better, higher P@5 is better)...")
    print(evaluate(movies, ratings).to_string())


def main(argv=None):
    ap = argparse.ArgumentParser(description="Movie recommender: collaborative + content-based filtering.")
    ap.add_argument("--movies", help="movies CSV (movieId,title,genres - MovieLens format)")
    ap.add_argument("--ratings", help="ratings CSV (userId,movieId,rating,...)")
    ap.add_argument("--user", type=int, help="recommend for this userId")
    ap.add_argument("--method", choices=METHODS, default="hybrid")
    ap.add_argument("--top", type=int, default=10, help="how many results to show")
    ap.add_argument("--similar", metavar="TITLE", help="movies similar to this title")
    ap.add_argument("--genres", help="comma-separated genres, e.g. 'Sci-Fi,Thriller'")
    ap.add_argument("--rate", metavar="'Title=5;Title=2'", help="rate movies as a brand-new user, get recommendations")
    ap.add_argument("--evaluate", action="store_true", help="hold-out evaluation")
    ap.add_argument("--repeats", type=int, default=3, help="random splits for --evaluate")
    ap.add_argument("--save-sample", metavar="DIR", help="write the sample data as MovieLens-style CSVs")
    args = ap.parse_args(argv)

    if bool(args.movies) != bool(args.ratings):
        ap.error("--movies and --ratings go together")
    if args.movies:
        movies, ratings, personas = pd.read_csv(args.movies), pd.read_csv(args.ratings), {}
    else:
        movies, ratings, personas = build_sample_data()
    if args.save_sample:
        from pathlib import Path
        Path(args.save_sample).mkdir(parents=True, exist_ok=True)
        movies.to_csv(Path(args.save_sample) / "movies.csv", index=False)
        ratings.to_csv(Path(args.save_sample) / "ratings.csv", index=False)
        print(f"Sample data written to {args.save_sample}")

    rec = Recommender(movies, ratings)
    acted = False
    try:
        if args.user is not None:
            show(f"User {args.user} - highest rated", rec.history(args.user))
            show(f"User {args.user} - {args.method} top {args.top}", rec.recommend(args.user, args.method, args.top))
            acted = True
        if args.similar:
            mid = rec.find_movie(args.similar)
            show(f"Similar to {rec.movies.loc[mid, 'title']} by genre", rec.similar_movies(mid, args.top, "content"))
            show(f"Similar to {rec.movies.loc[mid, 'title']} by rating patterns", rec.similar_movies(mid, args.top, "ratings"))
            acted = True
        if args.genres:
            show(f"Top picks for genres: {args.genres}", rec.by_genres([g.strip() for g in args.genres.split(",")], args.top))
            acted = True
        if args.rate:
            new_id = int(ratings["userId"].max()) + 1
            extra = pd.DataFrame([(new_id, m, r) for m, r in parse_ratings_arg(args.rate, rec)],
                                 columns=["userId", "movieId", "rating"])
            rec2 = Recommender(movies, pd.concat([ratings, extra], ignore_index=True))
            show("You rated", rec2.history(new_id, 20))
            show(f"Recommended for you ({args.method})", rec2.recommend(new_id, args.method, args.top))
            acted = True
        if args.evaluate:
            print(evaluate(movies, ratings, repeats=args.repeats).to_string())
            acted = True
        if not acted and not args.save_sample:
            run_demo(rec, movies, ratings, personas)
    except (ValueError, KeyError) as e:
        sys.exit(f"error: {e}")


if __name__ == "__main__":
    # Pass empty list to main to avoid parsing Colab's backend launch arguments
    main([])


Dataset: 42 users, 40 movies, 674 ratings (40% of the user x movie matrix filled)

User 1 (hidden taste group: action_fan) - highest rated so far
                         title                           genres  rating
           Interstellar (2014)           Adventure|Drama|Sci-Fi     5.0
Raiders of the Lost Ark (1981)                 Action|Adventure     5.0
             The Matrix (1999)                    Action|Sci-Fi     4.0
               Superbad (2007)                           Comedy     4.0
          Jurassic Park (1993) Action|Adventure|Sci-Fi|Thriller     4.0

User 1 - HYBRID top 5
                     title                  genres  predicted
          Inception (2010)  Action|Sci-Fi|Thriller       4.16
          Gladiator (2000)  Action|Adventure|Drama       4.09
 Mad Max: Fury Road (2015) Action|Adventure|Sci-Fi       4.02
              Alien (1979)           Horror|Sci-Fi       4.01
Saving Private Ryan (1998)        Action|Drama|War       3.86

User 1 - USER_CF top 3
   